In [1]:
import pandas as pd
import os

### Reading the data in and creating dataframes

In [2]:
# read all data in the data folder; will take ~4 minutes to run 

path = "data/"

print("Reading CSV")
print('2015...')
eug_cad2015 = pd.read_csv(os.path.join(path, "EugeneCAD2015noloc.csv"))
print('2016...')
eug_cad2016 = pd.read_csv(os.path.join(path, "EugeneCAD2016noloc.csv"))
print('2017...')
eug_cad2017 = pd.read_csv(os.path.join(path, "EugeneCAD2017noloc.csv"))
print('2018...')
eug_cad2018 = pd.read_csv(os.path.join(path, "EugeneCAD2018noloc.csv"))
print('2019...')
eug_cad2019 = pd.read_csv(os.path.join(path, "EugeneCAD2019noloc.csv"))
print('2020...')
eug_cad2020 = pd.read_csv(os.path.join(path, "EugeneCAD2020noloc.csv"))
print('2021...')
eug_cad2021 = pd.read_csv(os.path.join(path, "EugeneCAD2021noloc.csv"), low_memory=False)
print('2022...')
eug_cad2022 = pd.read_csv(os.path.join(path, "EugeneCAD2022noloc.csv"))
print('2023...')
eug_cad2023 = pd.read_csv(os.path.join(path, "EugeneCAD2023noloc.csv"))
print('2024...')
eug_cad2024 = pd.read_csv(os.path.join(path, "EugeneCAD2024noloc.csv"))
print('2025...')
eug_cad2025 = pd.read_csv(os.path.join(path, "EugeneCAD2025noloc.csv"), low_memory=False)

print("Reading Excel")
print('SPD Calls for Service...')
spd_calls = pd.concat(pd.read_excel('data/2015-2025 SPD Calls for Service.xlsx', sheet_name=None), ignore_index=True)
print('SPD Responding Units...')
spd_units = pd.concat(pd.read_excel('data/2015-2025 SPD Responding Units.xlsx', sheet_name=None), ignore_index=True)
print("MCSLC...")
mcslc = pd.concat(pd.read_excel('data/MCSLC.xlsx', sheet_name=None), ignore_index=True)
print("Done")

Reading CSV
2015...
2016...
2017...
2018...
2019...
2020...
2021...
2022...
2023...
2024...
2025...
Reading Excel
SPD Calls for Service...
SPD Responding Units...
MCSLC...
Done


In [3]:
dfs = [
    eug_cad2015, eug_cad2016, eug_cad2017, eug_cad2018,
    eug_cad2019, eug_cad2020, eug_cad2021, eug_cad2022,
    eug_cad2023, eug_cad2024, eug_cad2025
]

base_cols = set(dfs[0].columns)

for i, df in enumerate(dfs):
    cols = set(df.columns)
    missing = base_cols - cols
    extra = cols - base_cols
    
    if missing or extra:
        print(f"DataFrame {2015 + i}:")
        if missing:
            print("  Missing:", missing)
        if extra:
            print("  Extra:", extra)

# drop that extra column
eug_cad2025 = eug_cad2025.drop(columns={"month"})

DataFrame 2025:
  Extra: {'month'}


In [4]:
eug_cad = eug_cad2015
for df in dfs[1:]:
    eug_cad = pd.concat([eug_cad, df], ignore_index=True)


eug_cad.columns

eug_cad.head()

print(eug_cad.shape)

(1446014, 20)


### Cleaning Eugene CAD Dataset

In [5]:
eug_cad

,yr,agency,service,inci_id,calltime,case_id,callsource,nature,closecode,closed_as,secs_to_disp,secs_to_arrv,secs_to_close,disp,arrv,priority,primeunit,units_dispd,units_arrived,month
0,2015,EPD,LAW,15000001,2015-01-01 00:00:00.000,NaN,SELF,PERSON STOP,ASST,ASSISTED,0.0,0.0,217,1,1,6,_5E48,1,1,NaN
1,2015,EPD,LAW,15000002,2015-01-01 00:00:44.000,NaN,SELF,FIGHT,RSLV,RESOLVED,0.0,0.0,2114,1,1,P,_3F65,4,2,NaN
2,2015,EPD,LAW,15000003,2015-01-01 00:01:05.000,NaN,PHONE,CHECK WELFARE,ASST,ASSISTED,708.0,1094.0,1538,1,1,5,_3J79,1,1,NaN
3,2015,EPD,LAW,15000007,2015-01-01 00:03:16.000,NaN,W911,SHOTS FIRED,PCHK,PATROL CHECK,277.0,576.0,891,1,1,3,_5E48,2,2,NaN
4,2015,EPD,LAW,15000010,2015-01-01 00:03:34.000,NaN,SELF,ILLEGAL FIREWORKS,ADVI,ADVISED,0.0,0.0,150,1,1,5,_5K97,1,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446009,2025,EPD,LAW,25327368,2025-12-31 23:39:12.000,NaN,SELF,TRAFFIC STOP,SBCK,SOBRIETY CHECK,0.0,0.0,338,1,1,6,_4U41,1,1,12.0
1446010,2025,EPD,LAW,25327370,2025-12-31 23:40:48.000,NaN,SELF,TRAFFIC STOP,UTC,UNIFORM TRAFFIC CITATION ISSUED,1.0,1.0,478,1,1,6,_5E56,2,2,12.0
1446011,2025,EPD,LAW,25327374,2025-12-31 23:45:57.000,NaN,SELF,PATROL CHECK,UTC,UNIFORM TRAFFIC CITATION ISSUED,0.0,0.0,946,1,1,6,_4E48,3,3,12.0
1446012,2025,EPD,LAW,25327376,2025-12-31 23:49:12.000,NaN,SELF,ASSIST OREGON STATE POLICE,ASST,ASSISTED,0.0,0.0,765,1,1,P,_5T81,1,1,12.0


#### 1. Subset to the columns I need

In [6]:
columns = ["yr", "calltime", "nature", "closed_as", "primeunit"]
eug = eug_cad[columns].copy()

eug

,yr,calltime,nature,closed_as,primeunit
0,2015,2015-01-01 00:00:00.000,PERSON STOP,ASSISTED,_5E48
1,2015,2015-01-01 00:00:44.000,FIGHT,RESOLVED,_3F65
2,2015,2015-01-01 00:01:05.000,CHECK WELFARE,ASSISTED,_3J79
3,2015,2015-01-01 00:03:16.000,SHOTS FIRED,PATROL CHECK,_5E48
4,2015,2015-01-01 00:03:34.000,ILLEGAL FIREWORKS,ADVISED,_5K97
...,...,...,...,...,...
1446009,2025,2025-12-31 23:39:12.000,TRAFFIC STOP,SOBRIETY CHECK,_4U41
1446010,2025,2025-12-31 23:40:48.000,TRAFFIC STOP,UNIFORM TRAFFIC CITATION ISSUED,_5E56
1446011,2025,2025-12-31 23:45:57.000,PATROL CHECK,UNIFORM TRAFFIC CITATION ISSUED,_4E48
1446012,2025,2025-12-31 23:49:12.000,ASSIST OREGON STATE POLICE,ASSISTED,_5T81


#### 2. Turn ```calltime``` into a datetime object

In [7]:
eug["calltime"] = pd.to_datetime(eug["calltime"])
eug.drop('yr', axis=1, inplace=True)
eug.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1446014 entries, 0 to 1446013
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype         
---  ------     --------------    -----         
 0   calltime   1446014 non-null  datetime64[ns]
 1   nature     1445961 non-null  object        
 2   closed_as  1422086 non-null  object        
 3   primeunit  1091731 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 44.1+ MB


#### 3. Find CAHOOTS calls in Eugene

In [8]:
date_of_defunding = pd.to_datetime("2025-04-07")

eug_c = eug[(eug['primeunit'].str.contains('J', na=False)) & (eug['calltime'] < date_of_defunding)].copy()
eug_c

,calltime,nature,closed_as,primeunit
2,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,_3J79
17,2015-01-01 00:26:50,SUBJECT DOWN,ASSISTED,_3J79
27,2015-01-01 00:38:37,CHECK WELFARE,ASSISTED,_3J79
46,2015-01-01 01:26:17,ASSIST PUBLIC- POLICE,GONE ON ARRIVAL,_3J79
55,2015-01-01 01:41:40,CHECK WELFARE,ASSISTED,_3J79
...,...,...,...,...
1362841,2025-04-06 19:30:15,"PUBLIC ASSIST, CAHOOTS",GONE ON ARRIVAL,_4J79
1362852,2025-04-06 19:59:08,"PUBLIC ASSIST, CAHOOTS",ASSISTED,_4J79
1362858,2025-04-06 20:37:40,"PUBLIC ASSIST, CAHOOTS",ASSISTED,_4J79
1362859,2025-04-06 20:42:57,"PUBLIC ASSIST, CAHOOTS",ASSISTED,_4J79


#### 4. Find calls in Eugene after CAHOOTS was de-funded

In [9]:
eug_no_c = eug[(eug["calltime"] > date_of_defunding) & (~eug["primeunit"].str.contains('J', na=False))].copy()
eug_no_c["primeunit"] = eug_no_c["primeunit"].fillna("Unknown")

#### 5. Subset both datasets further to only have welfare check calls

In [10]:
eug_c = eug_c[eug_c["nature"] == "CHECK WELFARE"]
eug_c

,calltime,nature,closed_as,primeunit
2,2015-01-01 00:01:05,CHECK WELFARE,ASSISTED,_3J79
27,2015-01-01 00:38:37,CHECK WELFARE,ASSISTED,_3J79
55,2015-01-01 01:41:40,CHECK WELFARE,ASSISTED,_3J79
186,2015-01-01 15:09:21,CHECK WELFARE,ASSISTED,_3J79
427,2015-01-02 11:05:46,CHECK WELFARE,ASSISTED,_3J78
...,...,...,...,...
966708,2022-05-01 22:13:26,CHECK WELFARE,GONE ON ARRIVAL,_4J79
966735,2022-05-02 00:53:10,CHECK WELFARE,UNABLE TO LOCATE,_4J79
966748,2022-05-02 01:49:08,CHECK WELFARE,TRANSPORT MADE,_4J79
966749,2022-05-02 01:56:43,CHECK WELFARE,GONE ON ARRIVAL,_4J79


In [11]:
eug_no_c[eug_no_c["nature"] == "CHECK WELFARE"]

,calltime,nature,closed_as,primeunit
1362976,2025-04-07 09:49:15,CHECK WELFARE,QUALITY OF LIFE - NO DISPATCH,Unknown
1362978,2025-04-07 09:55:59,CHECK WELFARE,QUALITY OF LIFE - NO DISPATCH,Unknown
1363042,2025-04-07 12:42:09,CHECK WELFARE,REFERRED TO OTHER AGENCY,Unknown
1363043,2025-04-07 12:53:04,CHECK WELFARE,QUALITY OF LIFE - NO DISPATCH,Unknown
1363075,2025-04-07 14:34:14,CHECK WELFARE,REFERRED TO OTHER AGENCY,Unknown
...,...,...,...,...
1445916,2025-12-31 17:12:02,CHECK WELFARE,DISREGARDED BY DISPATCH,Unknown
1445927,2025-12-31 17:56:18,CHECK WELFARE,UNABLE TO LOCATE,_4U72
1445946,2025-12-31 20:23:13,CHECK WELFARE,WELFARE CHECK DONE,_5E56
1445957,2025-12-31 20:50:01,CHECK WELFARE,DISREGARD,_4U72


### Cleaning Springfield Call Dataset

In [12]:
spd_calls

,Incident Number,Initial Call Type,Final Call Type,Responding Agency,Primary Responding Unit,Call Creation Time,First Dispatched Time,First Arrival Time,Clear Time,Priority,Call Creation Mechanism
0,15000074,ALMAUD,AUDIBLE ALARM,SPD,1S18,2015-01-01 01:16:21,2015-01-01 01:18:28,2015-01-01 01:21:13,2015-01-01 01:32:08,3,PHONE
1,15000083,TRFSTP,DWS,SPD,3D2,2015-01-01 01:22:29,2015-01-01 01:22:29,2015-01-01 01:22:29,2015-01-01 01:34:57,4,SELF
2,15000167,MVAUNK,MOTOR VEH ACC UNKNOWN INJ,SPD,,2015-01-01 03:15:28,NaT,NaT,NaT,3,E911
3,15000216,TRFSTP,DWS,SPD,1S22,2015-01-01 05:27:24,2015-01-01 05:27:24,2015-01-01 05:27:24,2015-01-01 05:38:53,4,SELF
4,15000222,SUSPVE,SUSPICIOUS VEHICLE,SPD,1S11,2015-01-01 05:52:53,2015-01-01 05:53:57,2015-01-01 05:56:21,2015-01-01 06:04:04,3,PHONE
...,...,...,...,...,...,...,...,...,...,...,...
567650,25326475,DISVEH,DISABLED VEHICLE,SPD,1S23,2025-12-30 21:15:10,2025-12-30 21:15:10,2025-12-30 21:15:10,2025-12-30 21:17:20,4,SELF
567651,25326480,TRFSTP,TRAFFIC STOP,SPD,1S23,2025-12-30 21:25:51,2025-12-30 21:25:51,2025-12-30 21:25:51,2025-12-30 21:30:21,6,SELF
567652,25326519,CHKWLF,CHECK WELFARE,SPD,3J81,2025-12-30 22:24:34,2025-12-30 22:25:32,2025-12-30 22:31:12,2025-12-30 22:47:29,7,PHONE
567653,25326528,ALMAUD,AUDIBLE ALARM,SPD,1S21,2025-12-30 22:34:54,2025-12-30 22:41:36,2025-12-30 22:44:28,2025-12-30 22:48:25,3,PHONE


#### 1. Subset to columns I need and rename

In [13]:
columns = ["Final Call Type", "Call Creation Time", "Primary Responding Unit"]
spd = spd_calls[columns].copy()

spd.rename(columns={
    "Final Call Type": "nature",
    "Call Creation Time": "calltime",
    "Primary Responding Unit": "primeunit"
}, inplace=True)

spd

,nature,calltime,primeunit
0,AUDIBLE ALARM,2015-01-01 01:16:21,1S18
1,DWS,2015-01-01 01:22:29,3D2
2,MOTOR VEH ACC UNKNOWN INJ,2015-01-01 03:15:28,
3,DWS,2015-01-01 05:27:24,1S22
4,SUSPICIOUS VEHICLE,2015-01-01 05:52:53,1S11
...,...,...,...
567650,DISABLED VEHICLE,2025-12-30 21:15:10,1S23
567651,TRAFFIC STOP,2025-12-30 21:25:51,1S23
567652,CHECK WELFARE,2025-12-30 22:24:34,3J81
567653,AUDIBLE ALARM,2025-12-30 22:34:54,1S21


#### 2. Turn ```calltime``` into a datetime object

In [14]:
spd["calltime"] = pd.to_datetime(spd["calltime"])
spd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 567655 entries, 0 to 567654
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   nature     567632 non-null  object        
 1   calltime   567655 non-null  datetime64[ns]
 2   primeunit  567655 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 13.0+ MB


#### 3. Clean the ```primeunit``` column

In [15]:
spd["primeunit"] = spd["primeunit"].str.strip()

### CSV Output

In [16]:
eug_no_c.to_csv("cleaned_eug_no_c.csv", index=False)
eug_c.to_csv("cleaned_eug_c.csv", index=False)
spd.to_csv("cleaned_spd.csv", index=False)